<a href="https://colab.research.google.com/github/jihandong/MLCExercises/blob/main/mlc_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Machine Learning Compiler Exercises

## Prepare the Environment




In [1]:
%pip install -q "apache-tvm==0.26.0" "apache-tvm-ffi==0.1.16"

In [2]:
import numpy as np
import tvm
from tvm.script import tirx as T

dtype = "float32"
a_np = np.random.rand(128, 128).astype(dtype)
b_np = np.random.rand(128, 128).astype(dtype)
c_mm_relu = np.maximum(a_np @ b_np, 0)

@tvm.script.ir_module
class MyModule:
    @T.prim_func(s_tir=True)
    def mm_relu(A: T.Buffer((128, 128), "float32"),
                B: T.Buffer((128, 128), "float32"),
                C: T.Buffer((128, 128), "float32")):
        T.func_attr({"global_symbol": "mm_relu", "tirx.noalias": True})
        Y = T.sblock_alloc_buffer((128, 128), dtype="float32")
        for i, j, k in T.grid(128, 128, 128):
            with T.sblock("Y"):
                vi, vj, vk = T.axis.remap("SSR", [i, j, k])
                with T.init():
                    Y[vi, vj] = T.float32(0)
                Y[vi, vj] = Y[vi, vj] + A[vi, vk] * B[vk, vj]
        for i, j in T.grid(128, 128):
            with T.sblock("C"):
                vi, vj = T.axis.remap("SS", [i, j])
                C[vi, vj] = T.max(Y[vi, vj], T.float32(0))


## 2.4.5.1 J1 Profile

In [3]:
from statistics import median


def transform(mod, j_factor):
    """Apply the loop transform from section 2.4.5.1 to a fresh schedule."""
    schedule = tvm.s_tir.Schedule(mod)
    matmul = schedule.get_sblock("Y", func_name="mm_relu")
    relu = schedule.get_sblock("C", func_name="mm_relu")
    _, j, k = schedule.get_loops(matmul)
    j_outer, j_inner = schedule.split(j, factors=[None, j_factor])
    schedule.reorder(j_outer, k, j_inner)
    schedule.reverse_compute_at(relu, j_outer)
    return schedule.mod


dev = tvm.cpu(0)
a_nd = tvm.runtime.tensor(a_np, device=dev)
b_nd = tvm.runtime.tensor(b_np, device=dev)
c_nd = tvm.runtime.empty(c_mm_relu.shape, dtype, dev)


def profile_ms(mod):
    """Return the median time per kernel call in milliseconds."""
    lib = tvm.build(mod, target="llvm")
    timer = lib.time_evaluator("mm_relu", dev, number=100, repeat=5)
    return median(timer(a_nd, b_nd, c_nd).results) * 1_000


In [6]:
j_factors = [1,2,4,8,16,32,64,128]

baseline_ms = profile_ms(MyModule)
print(f"{'j_factor':>10} {'median (ms)':>14} {'speedup':>10}")
print(f"{'baseline':>10} {baseline_ms:14.4f} {'1.00x':>10}")

for j_factor in j_factors:
    scheduled = transform(MyModule, j_factor)
    elapsed_ms = profile_ms(scheduled)
    print(f"{j_factor:10d} {elapsed_ms:14.4f} {baseline_ms / elapsed_ms:9.2f}x")


  j_factor    median (ms)    speedup
  baseline         3.7445      1.00x
         1         2.9627      1.26x
         2         1.7178      2.18x
         4         0.7911      4.73x
         8         0.4841      7.73x
        16         0.3321     11.28x
        32         0.2881     13.00x
        64         0.3100     12.08x
       128         0.3085     12.14x
